# What-If Scenario Planning — Coles ANZ (Hair Care & Shave Care)

This notebook is a **fully self-contained** What-If Scenario Planning system adapted for
**Coles Australia** using real historical baseline estimation data.

## What this notebook covers

| Step | Description |
|---|---|
| **1. Data Loading** | Load `historical_baseline_estimation.xlsx` and map to scenario-ready format |
| **2. Data Models** | Pydantic models for promotion validation (adapted from TPO TAL schema) |
| **3. Scenario Operations** | ScenarioStore, filter, swap_replace, concatenate |
| **4. Mock Scoring** | Heuristic scoring engine using Coles/ANZ brand parameters |
| **5. Comparison & Viz** | Compare scenarios, calculate lift %, Plotly dashboards |
| **6. End-to-End Example** | Full walk-through with Herbal Essences & Pantene scenarios |

---

### Data Source

- **File**: `data/historical_baseline_estimation.xlsx`
- **Market**: Coles / ANZ
- **Categories**: Hair Care, Shave Care
- **Brands**: Herbal Essences, Head & Shoulders, Pantene, Gillette, Venus
- **Granularity**: SKU × Week (189 SKUs, ~86 weeks)
- **Promo mechanics**: 50% Straight, 30% off, 40% MB, Del/Excl, etc.

## Setup — Install & Import

In [1]:
# Install dependencies if needed
# !pip install pydantic pandas plotly openpyxl numpy

In [2]:
import json
import re
import copy
from typing import Dict, List, Literal, Optional, Any
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pydantic import BaseModel, ConfigDict, Field, field_validator, model_validator

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.2f}'.format)

print("\u2705 Imports OK")

✅ Imports OK


---
## Part 1 — Load & Transform Real Data

The `historical_baseline_estimation.xlsx` file uses a **Coles/ANZ schema** with columns like
`promo_mechanic`, `promo_depth`, `baseline_sales_qty`, etc.

We load this data and map it into a scenario-ready format that the downstream
tools (filter, swap_replace, scoring) can work with.

In [3]:
# ============================================================
# LOAD REAL DATA
# ============================================================
DATA_PATH = Path("../data/historical_baseline_estimation.xlsx")
raw_df = pd.read_excel(DATA_PATH)

print(f"Loaded {len(raw_df):,} rows \u00d7 {len(raw_df.columns)} columns")
print(f"Date range: {raw_df['start_date'].min()} to {raw_df['start_date'].max()}")
print(f"\nBrands: {raw_df['brand'].unique().tolist()}")
print(f"Categories: {raw_df['category_name'].unique().tolist()}")
print(f"SKUs: {raw_df['sku_name'].nunique()}")
print(f"\nPromo flag distribution:")
print(raw_df['promo_flag'].value_counts())
print(f"\nTop promo mechanics:")
print(raw_df['promo_mechanic'].value_counts().head(10))
raw_df.head(3)

Loaded 12,626 rows × 25 columns
Date range: 2024-07-03 to 2026-02-25

Brands: ['HERBAL ESSENCES', 'HEAD & SHOULDERS', 'PANTENE', 'GILLETTE', 'VENUS']
Categories: ['HAIR CARE', 'SHAVE CARE']
SKUs: 189

Promo flag distribution:
promo_flag
True     8120
False    4506
Name: count, dtype: int64

Top promo mechanics:
promo_mechanic
Baseline            4506
30% off             1096
Del/Excl            1023
50% Straight         778
50% OFF              481
50% off              431
40% MB               368
10% off              333
Summer Base / FB     328
Summer Baseline      312
Name: count, dtype: int64


,country_name,customer_name,category_name,sku_code,brand,segment,sku_name,start_date,month,quarter,year,end_date,promo_flag,promo_mechanic,promo_depth,non_promo_price,promo_price,actual_sales_qty,baseline_sales_qty,incremental_qty,promo_revenue,baseline_revenue,incremental_revenue,cost_of_promotion,actual_promo_cost
0,ANZ,COLES,HAIR CARE,190679000000.00,HERBAL ESSENCES,Hair Conditioner,HE CN ARGAN OIL 4 20.2Z 600ML,2024-07-03,7,3,2024,2024-07-09,True,50% Straight,1.00,19.00,10.00,3770,472.00,3298.00,35815.00,8971.00,26844.00,4.00,15796.00
1,ANZ,COLES,HAIR CARE,190679000000.00,HERBAL ESSENCES,Hair Conditioner,HE CN ARGAN OIL 4 20.2Z 600ML,2024-07-10,7,3,2024,2024-07-16,True,32% MB,0.00,19.00,13.00,1912,472.00,1440.00,24703.00,8971.00,15732.00,2.00,3977.00
2,ANZ,COLES,HAIR CARE,190679000000.00,HERBAL ESSENCES,Hair Conditioner,HE CN ARGAN OIL 4 20.2Z 600ML,2024-07-17,7,3,2024,2024-07-23,True,30% Straight,0.00,19.00,13.00,1802,472.00,1330.00,23967.00,8971.00,14996.00,3.00,4541.00


In [4]:
# ============================================================
# MAP COLES SCHEMA TO SCENARIO-READY FORMAT
# ============================================================
# The Coles data uses promo_mechanic strings like "50% Straight", "30% MB", etc.
# We map these to a unified format that the scenario operations can work with.

def map_promo_mechanic(mechanic: str, depth: float, non_promo_price: float) -> dict:
    """Map a Coles promo_mechanic string to TAL-like fields."""
    result = {
        "promo_type": "Baseline",
        "discount_pct": 0.0,
        "is_multibuy": 0,
    }
    if not mechanic or mechanic == "Baseline":
        return result

    m_lower = mechanic.lower().strip()

    # Extract percentage from mechanic string
    pct_match = re.search(r'(\d+)%', mechanic)
    pct = float(pct_match.group(1)) / 100 if pct_match else depth

    if 'mb' in m_lower or 'multibuy' in m_lower:
        result["promo_type"] = "Multibuy"
        result["discount_pct"] = pct
        result["is_multibuy"] = 1
    elif 'straight' in m_lower or 'off' in m_lower:
        result["promo_type"] = "Discount"
        result["discount_pct"] = pct
    elif 'edv' in m_lower:
        result["promo_type"] = "EDV"
        result["discount_pct"] = pct
    elif 'del' in m_lower or 'excl' in m_lower:
        result["promo_type"] = "Delisted/Exclusive"
        result["discount_pct"] = pct
    elif 'fb' in m_lower or 'base' in m_lower:
        result["promo_type"] = "Feature/Base"
        result["discount_pct"] = pct
    elif 'oge' in m_lower:
        result["promo_type"] = "OGE"
        result["discount_pct"] = pct
    elif 'd+l' in m_lower or 'd&l' in m_lower:
        result["promo_type"] = "Display+Leaflet"
        result["discount_pct"] = pct
    else:
        result["promo_type"] = mechanic
        result["discount_pct"] = pct

    return result


def transform_to_scenario_format(df: pd.DataFrame) -> pd.DataFrame:
    """Transform Coles baseline data into scenario-ready format."""
    out = df.copy()

    # Rename columns to scenario format
    out = out.rename(columns={
        "sku_code": "SKU ID",
        "sku_name": "SKU Name",
        "brand": "Brand",
        "segment": "Segment",
        "category_name": "Category",
        "customer_name": "Customer",
        "country_name": "Country",
        "start_date": "Fiscal Week Start Date",
        "end_date": "Fiscal Week End Date",
        "baseline_sales_qty": "Base Consumption Units",
        "actual_sales_qty": "Actual Units",
        "promo_flag": "promo_flag",
        "promo_mechanic": "Promo Mechanic",
        "promo_depth": "Promo Depth",
        "non_promo_price": "Non Promo Price",
        "promo_price": "Promo Price",
        "incremental_qty": "Incremental Units",
        "promo_revenue": "Promo Revenue",
        "baseline_revenue": "Baseline Revenue",
        "incremental_revenue": "Incremental Revenue",
        "cost_of_promotion": "Cost of Promotion",
        "actual_promo_cost": "Actual Promo Cost",
    })

    # Map promo mechanics to structured fields
    promo_info = out.apply(
        lambda r: map_promo_mechanic(
            r["Promo Mechanic"], r["Promo Depth"], r["Non Promo Price"]
        ), axis=1, result_type="expand"
    )
    out["Promo Type"] = promo_info["promo_type"]
    out["Discount Pct"] = promo_info["discount_pct"]
    out["Is Multibuy"] = promo_info["is_multibuy"]

    # Ensure date strings
    out["Fiscal Week Start Date"] = pd.to_datetime(out["Fiscal Week Start Date"]).dt.strftime("%Y-%m-%d")
    out["Fiscal Week End Date"] = pd.to_datetime(out["Fiscal Week End Date"]).dt.strftime("%Y-%m-%d")

    return out


base_df = transform_to_scenario_format(raw_df)
print(f"Transformed to {len(base_df):,} rows \u00d7 {len(base_df.columns)} columns")
print(f"\nPromo type distribution:")
print(base_df["Promo Type"].value_counts())
base_df[["Brand", "SKU Name", "Fiscal Week Start Date", "Promo Mechanic",
         "Promo Type", "Discount Pct", "Base Consumption Units", "Actual Units"]].head(5)

Transformed to 12,626 rows × 28 columns

Promo type distribution:
Promo Type
Baseline               4506
Discount               3827
Delisted/Exclusive     1023
Feature/Base            918
Multibuy                666
Summer 30%              264
OGE                     156
Summer 50%              140
Display+Leaflet         134
EDV                      95
2 FOR $24                89
50% CLEARANCE            83
2 FOR $21                76
40% CLEARANCE            74
2 FOR $36                72
2 FOR $17                72
DD 300ml                 56
2 FOR $16                42
Coles upgrade            41
2 FOR $12 PTN            40
30% End Display          40
50% Top End Display      35
2 FOR $12 H.E            34
50% End Display          32
50% FGE                  30
50% Multi Buy            27
30% Multi Buy            22
2 FOR $20                18
PRICE ESTAB.             12
2 FOR $28                 1
40% End Display           1
Name: count, dtype: int64


,Brand,SKU Name,Fiscal Week Start Date,Promo Mechanic,Promo Type,Discount Pct,Base Consumption Units,Actual Units
0,HERBAL ESSENCES,HE CN ARGAN OIL 4 20.2Z 600ML,2024-07-03,50% Straight,Discount,0.50,472.00,3770
1,HERBAL ESSENCES,HE CN ARGAN OIL 4 20.2Z 600ML,2024-07-10,32% MB,Multibuy,0.32,472.00,1912
2,HERBAL ESSENCES,HE CN ARGAN OIL 4 20.2Z 600ML,2024-07-17,30% Straight,Discount,0.30,472.00,1802
3,HERBAL ESSENCES,HE CN ARGAN OIL 4 20.2Z 600ML,2024-07-24,Baseline,Baseline,0.00,472.00,475
4,HERBAL ESSENCES,HE CN ARGAN OIL 4 20.2Z 600ML,2024-07-31,50% Straight,Discount,0.50,472.00,4607


---
## Part 2 — Promotion Data Model

A simplified Pydantic model adapted for Coles/ANZ promo mechanics.
Instead of the full TAL schema (CRL/TPR/Coupon/FeatureDisplay),
we use the promo types found in the actual data:
- **Discount** — percentage off (e.g., 30% Straight, 50% OFF)
- **Multibuy** — buy multiple for discount (e.g., 30% MB, 40% MB)
- **EDV** — Every Day Value pricing
- **Feature/Base** — feature ad or summer base pricing
- **Display+Leaflet** — in-store display with leaflet
- **OGE** — On-Going Everyday promotion

In [5]:
# ============================================================
# COLES PROMOTION DATA MODEL
# ============================================================

VALID_PROMO_TYPES = [
    "Baseline", "Discount", "Multibuy", "EDV",
    "Feature/Base", "Display+Leaflet", "OGE",
    "Delisted/Exclusive", "Other",
]


class ColesPromotion(BaseModel):
    """Promotion model for Coles ANZ promo mechanics."""
    model_config = ConfigDict(populate_by_name=True)

    promo_type: str = Field(default="Baseline", alias="Promo Type")
    discount_pct: float = Field(default=0.0, ge=0.0, le=1.0, alias="Discount Pct")
    is_multibuy: Literal[0, 1] = Field(default=0, alias="Is Multibuy")
    promo_mechanic: str = Field(default="Baseline", alias="Promo Mechanic")
    promo_depth: float = Field(default=0.0, ge=0.0, alias="Promo Depth")

    @field_validator("is_multibuy", mode="before")
    @classmethod
    def convert_multibuy(cls, v):
        if isinstance(v, bool): return 1 if v else 0
        if isinstance(v, int): return 1 if v else 0
        return 0

    @model_validator(mode="after")
    def sync_flags(self):
        if self.promo_type == "Multibuy":
            self.is_multibuy = 1
        if self.promo_type == "Baseline":
            self.discount_pct = 0.0
            self.is_multibuy = 0
        return self


# Column name mapping for validation
_PROMO_COL_TO_ATTR = {}
_PROMO_ATTR_TO_COL = {}
for _name, _fi in ColesPromotion.model_fields.items():
    _col = _fi.alias or _name
    _PROMO_COL_TO_ATTR[_col] = _name
    _PROMO_ATTR_TO_COL[_name] = _col

_PROMO_FIELDS = set(_PROMO_COL_TO_ATTR)


# Quick test
p = ColesPromotion(promo_type="Discount", discount_pct=0.30, promo_mechanic="30% Straight")
print(f"Discount promo: type={p.promo_type}, pct={p.discount_pct}, mb={p.is_multibuy}")

p2 = ColesPromotion(promo_type="Multibuy", discount_pct=0.40, promo_mechanic="40% MB")
print(f"Multibuy promo: type={p2.promo_type}, pct={p2.discount_pct}, mb={p2.is_multibuy}")
print("\u2705 Promotion model OK")

Discount promo: type=Discount, pct=0.3, mb=0
Multibuy promo: type=Multibuy, pct=0.4, mb=1
✅ Promotion model OK


---
## Part 3 — Scenario Operations

All operations work on pandas DataFrames held in an in-memory `ScenarioStore`.

### Key Operations
- **`copy_base`** — Clone base data before any edits
- **`col_filter`** — Filter to specific brand/segment/SKU
- **`swap_replace`** — Apply new promo values to filtered rows, merge back with base
- **`concatenate_scenarios`** — Combine multiple scenarios for batch scoring

In [6]:
# ============================================================
# SCENARIO STORE (in-memory)
# ============================================================

class ScenarioStore:
    """Simple in-memory store for scenario DataFrames."""

    def __init__(self):
        self._store: Dict[str, pd.DataFrame] = {}

    def save(self, key: str, df: pd.DataFrame) -> str:
        self._store[key] = df.copy()
        return key

    def load(self, key: str) -> pd.DataFrame:
        if key not in self._store:
            raise KeyError(f"No scenario '{key}' in store. Available: {list(self._store.keys())}")
        return self._store[key].copy()

    def list_keys(self) -> List[str]:
        return list(self._store.keys())

    def __repr__(self):
        return f"ScenarioStore({list(self._store.keys())})"


print("\u2705 ScenarioStore ready")

✅ ScenarioStore ready


In [7]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def validated_promotion_updates(promotion_values: dict, df_columns: set) -> dict:
    """Validate promotion values against the Pydantic model before writing."""
    if not isinstance(promotion_values, dict) or not promotion_values:
        raise ValueError("promotion_values must be a non-empty dict.")

    unknown = sorted(set(promotion_values) - _PROMO_FIELDS)
    if unknown:
        raise ValueError(f"Unknown promotion fields: {unknown}")

    attr_values = {_PROMO_COL_TO_ATTR.get(k, k): v for k, v in promotion_values.items()}
    validated = ColesPromotion(**attr_values)

    fields_to_apply = set(promotion_values)
    missing = sorted(f for f in fields_to_apply if f not in df_columns)
    if missing:
        print(f"[WARN] Skipping fields not in data: {missing}")
        fields_to_apply -= set(missing)

    attr_include = {_PROMO_COL_TO_ATTR.get(f, f) for f in fields_to_apply}
    return validated.model_dump(by_alias=True, include=attr_include, exclude_none=False)


def filter_zero_base(df: pd.DataFrame) -> pd.DataFrame:
    """Remove rows where Base Consumption Units <= 0."""
    col = "Base Consumption Units"
    if col in df.columns:
        return df[df[col] > 0].reset_index(drop=True)
    return df


def generate_scenario_slug(filter_criteria: dict, promo_desc: str) -> str:
    """Auto-generate a short scenario name."""
    parts = []
    for val in filter_criteria.values():
        if val:
            clean = re.sub(r"[^a-z0-9_]", "", str(val).lower().replace(" ", "_"))
            parts.append(clean[:15])
    promo = promo_desc.lower()
    if m := re.search(r'(\d+)%', promo):
        parts.append(f"{m.group(1)}pct")
    if 'mb' in promo or 'multibuy' in promo:
        parts.append("mb")
    if 'straight' in promo:
        parts.append("str")
    return "_".join(parts or ["scenario"])[:50]


print("\u2705 Helper functions OK")

✅ Helper functions OK


In [8]:
# ============================================================
# SCENARIO OPERATION FUNCTIONS
# ============================================================

# Dedup key: SKU ID + Week
DEDUP_COLS = ["SKU ID", "Fiscal Week Start Date"]


def copy_base(store: ScenarioStore, source_key: str, dest_key: str = "base") -> str:
    """Clone the source scenario into a new key."""
    df = store.load(source_key)
    return store.save(dest_key, df)


def get_cols_and_values(store: ScenarioStore, key: str,
                        columns: Optional[List[str]] = None,
                        max_values: int = 25) -> str:
    """Return JSON summary of column unique values for exploration."""
    df = store.load(key)
    default_cols = ["Brand", "Segment", "Category", "Fiscal Week Start Date"]
    cols = [c for c in (columns or default_cols) if c in df.columns]
    result = {}
    for col in cols:
        vals = df[col].dropna().unique()
        if len(vals) > max_values:
            result[col] = f"({len(vals)} unique values)"
        else:
            result[col] = [v.item() if hasattr(v, "item") else v for v in vals]
    return json.dumps(result, indent=2, default=str)


def col_filter(store: ScenarioStore, source_key: str, filter_criteria: dict,
               dest_key: Optional[str] = None) -> str:
    """Filter rows by column=value criteria."""
    df = store.load(source_key)
    for col, val in filter_criteria.items():
        df = df[df[col] == val]
        if df.empty:
            raise ValueError(f"No rows for {col}='{val}'")
    out_key = dest_key or f"{source_key}_filtered"
    return store.save(out_key, df.reset_index(drop=True))


def swap_replace(store: ScenarioStore,
                 filtered_key: str,
                 base_key: str,
                 promotion_values: dict,
                 dest_key: Optional[str] = None) -> str:
    """Apply promotion values to filtered rows, merge back with base.

    1. Load filtered rows
    2. Validate & apply promotion_values
    3. Concatenate with base; deduplicate (edited rows win)
    4. Save under dest_key
    """
    edited = store.load(filtered_key)
    base = store.load(base_key)

    updates = validated_promotion_updates(promotion_values, set(edited.columns))
    for col, val in updates.items():
        edited[col] = val

    # Also update the promo_flag to True for edited rows
    edited["promo_flag"] = True

    combined = pd.concat([base, edited], ignore_index=True)
    combined = combined.drop_duplicates(subset=DEDUP_COLS, keep="last")

    out_key = dest_key or f"{filtered_key}_scenario"
    return store.save(out_key, combined)


def rename_scenario(store: ScenarioStore, key: str, scenario_name: str) -> str:
    """Set the 'Scenario Name' column."""
    df = store.load(key)
    df["Scenario Name"] = scenario_name
    return store.save(key, df)


def concatenate_scenarios(store: ScenarioStore,
                          scenario_keys: List[str],
                          base_key: str,
                          combined_key: str = "all_scenarios",
                          include_base: bool = True) -> str:
    """Merge multiple named scenarios (+ base) into one DataFrame."""
    dfs = []
    if include_base:
        base_df = store.load(base_key)
        base_df["Scenario Name"] = "base"
        dfs.append(base_df)

    for k in scenario_keys:
        df = store.load(k)
        dfs.append(df)

    combined = pd.concat(dfs, ignore_index=True)
    combined = combined.drop_duplicates(
        subset=DEDUP_COLS + ["Scenario Name"], keep="last"
    )
    return store.save(combined_key, combined)


def list_scenarios_in_df(store: ScenarioStore, key: str) -> None:
    """Print scenario names and row counts."""
    df = store.load(key)
    if "Scenario Name" not in df.columns:
        print("No 'Scenario Name' column found.")
        return
    counts = df.groupby("Scenario Name").size().rename("rows")
    print(counts.to_string())


print("\u2705 Scenario operation functions OK")

✅ Scenario operation functions OK


---
## Part 4 — Mock Scoring Engine

Since we don't have access to the SB+ ML API, we simulate scoring with a
**heuristic uplift model** adapted for Coles/ANZ brands.

The model:
1. Reads `Base Consumption Units` (baseline sales from DLM-MSCM)
2. Applies promo uplift based on the `Promo Type` and `Discount Pct`
3. Generates `Pred_Units`, `Incr_Units`, `GIV`, `NOS`, and spend metrics

In [9]:
# ============================================================
# MOCK SCORING ENGINE (Coles/ANZ)
# ============================================================

# Average retail prices and margins per brand (AUD)
BRAND_PARAMS = {
    "HERBAL ESSENCES": {"price": 15.00, "gm": 0.45, "nos_margin": 0.30},
    "HEAD & SHOULDERS": {"price": 14.00, "gm": 0.48, "nos_margin": 0.32},
    "PANTENE":          {"price": 16.00, "gm": 0.47, "nos_margin": 0.31},
    "GILLETTE":         {"price": 25.00, "gm": 0.55, "nos_margin": 0.38},
    "VENUS":            {"price": 20.00, "gm": 0.52, "nos_margin": 0.35},
}
DEFAULT_PARAMS = {"price": 15.00, "gm": 0.45, "nos_margin": 0.30}


def _compute_row_uplift(row: pd.Series) -> float:
    """Calculate promotional uplift multiplier for a single row."""
    uplift = 1.0
    promo_type = row.get("Promo Type", "Baseline")
    discount_pct = float(row.get("Discount Pct", 0))
    is_mb = int(row.get("Is Multibuy", 0))

    if promo_type == "Baseline" or discount_pct == 0:
        return 1.0

    # Discount (straight % off)
    if promo_type == "Discount":
        # Price elasticity: higher discount = larger uplift, with diminishing returns
        uplift += min(discount_pct * 2.5, 0.80)  # cap at +80%

    # Multibuy (buy X get Y% off)
    elif promo_type == "Multibuy":
        # Multibuys are slightly less effective per-unit than straight discounts
        uplift += min(discount_pct * 2.0, 0.65)  # cap at +65%

    # EDV (everyday value)
    elif promo_type == "EDV":
        uplift += min(discount_pct * 1.5, 0.30)  # modest but sustained

    # Feature/Base (feature ad pricing)
    elif promo_type == "Feature/Base":
        uplift += min(discount_pct * 1.8, 0.40)

    # Display+Leaflet
    elif promo_type == "Display+Leaflet":
        uplift += min(discount_pct * 2.2, 0.55)

    # OGE
    elif promo_type == "OGE":
        uplift += min(discount_pct * 1.8, 0.40)

    # Other / catch-all
    else:
        uplift += min(discount_pct * 2.0, 0.50)

    return uplift


def mock_score_scenarios(store: ScenarioStore, key: str,
                         scored_key: str = "scored") -> str:
    """Simulate ML scoring. Adds Pred_Units, Incr_Units, GIV, NOS."""
    df = store.load(key).copy()

    if "Base Consumption Units" not in df.columns:
        raise ValueError("DataFrame must have 'Base Consumption Units' column.")

    uplifts = df.apply(_compute_row_uplift, axis=1)
    df["Pred_Units"] = (df["Base Consumption Units"] * uplifts).round(0).astype(int)
    df["Incr_Units"] = (df["Pred_Units"] - df["Base Consumption Units"]).clip(lower=0)

    # Price and margin lookup
    df["_price"] = df["Brand"].map({b: p["price"] for b, p in BRAND_PARAMS.items()}).fillna(DEFAULT_PARAMS["price"])
    df["_gm"]    = df["Brand"].map({b: p["gm"] for b, p in BRAND_PARAMS.items()}).fillna(DEFAULT_PARAMS["gm"])
    df["_nos_m"] = df["Brand"].map({b: p["nos_margin"] for b, p in BRAND_PARAMS.items()}).fillna(DEFAULT_PARAMS["nos_margin"])

    # Use actual Non Promo Price if available, otherwise brand average
    if "Non Promo Price" in df.columns:
        actual_price = df["Non Promo Price"].where(df["Non Promo Price"] > 0, df["_price"])
    else:
        actual_price = df["_price"]

    df["GIV"]       = (df["Pred_Units"] * actual_price).round(2)
    df["Incr_GIV"]  = (df["Incr_Units"] * actual_price).round(2)
    df["NOS"]       = (df["GIV"] * df["_nos_m"]).round(2)
    df["Incr_NOS"]  = (df["Incr_GIV"] * df["_nos_m"]).round(2)

    # Estimated promo spend = discount amount * predicted units
    discount_pct = df.get("Discount Pct", pd.Series(0, index=df.index)).fillna(0)
    df["EstimatedPromoSpend"] = (discount_pct * actual_price * df["Pred_Units"]).round(2)

    df.drop(columns=["_price", "_gm", "_nos_m"], inplace=True)

    print(f"Scored {len(df):,} rows across scenarios: {df['Scenario Name'].unique().tolist()}")
    return store.save(scored_key, df)


print("\u2705 Mock scoring engine OK")

✅ Mock scoring engine OK


---
## Part 5 — Comparison & Visualization

Functions to aggregate, compare, and plot KPIs across all scenarios.

In [10]:
# ============================================================
# METRIC DEFINITIONS
# ============================================================

SUM_METRICS = [
    "Base Consumption Units", "Pred_Units", "Incr_Units",
    "GIV", "Incr_GIV", "NOS", "Incr_NOS",
    "EstimatedPromoSpend",
]

KEY_METRICS = ["Pred_Units", "Incr_Units", "GIV", "NOS"]


# ============================================================
# COMPARISON FUNCTIONS
# ============================================================

def compare_scenarios(store: ScenarioStore, key: str,
                      metrics: Optional[List[str]] = None) -> pd.DataFrame:
    """Aggregate KPIs by Scenario Name."""
    df = store.load(key)
    if "Scenario Name" not in df.columns:
        raise ValueError("No 'Scenario Name' column.")

    use_metrics = metrics or SUM_METRICS
    sum_cols = [m for m in use_metrics if m in df.columns]
    summary = df.groupby("Scenario Name")[sum_cols].sum().reset_index()
    for col in sum_cols:
        summary[col] = summary[col].round(2)
    return summary


def get_scenario_lift_summary(store: ScenarioStore, key: str) -> pd.DataFrame:
    """Calculate % lift vs base for key metrics."""
    df = store.load(key)
    lift_metrics = [m for m in KEY_METRICS if m in df.columns]
    summary = df.groupby("Scenario Name")[lift_metrics].sum().reset_index()

    base_row = summary[summary["Scenario Name"] == "base"]
    if base_row.empty:
        raise ValueError("No 'base' scenario found.")

    base_vals = base_row[lift_metrics].iloc[0]
    rows = []
    for _, row in summary.iterrows():
        if row["Scenario Name"] == "base":
            continue
        lift = {"Scenario Name": row["Scenario Name"]}
        for m in lift_metrics:
            bv = base_vals[m]
            lift[f"{m} Lift %"] = round((row[m] - bv) / bv * 100, 2) if bv else 0.0
        rows.append(lift)
    return pd.DataFrame(rows)


# ============================================================
# VISUALIZATION FUNCTIONS
# ============================================================

def plot_scenario_comparison(store: ScenarioStore, key: str,
                             metric: str = "Pred_Units",
                             title: Optional[str] = None) -> go.Figure:
    """Bar chart comparing scenarios on a single metric."""
    df = store.load(key)
    summary = df.groupby("Scenario Name")[metric].sum().reset_index()
    summary = summary.sort_values(metric, ascending=False)

    colors = ["#7f7f7f" if n == "base" else "#1f77b4" for n in summary["Scenario Name"]]

    fig = go.Figure(go.Bar(
        x=summary["Scenario Name"], y=summary[metric],
        marker_color=colors,
        text=summary[metric].round(0),
        textposition="outside",
    ))
    fig.update_layout(
        title=title or f"Scenario Comparison: {metric}",
        xaxis_title="Scenario", yaxis_title=metric,
        template="plotly_white", height=450,
    )
    return fig


def plot_scenario_dashboard(store: ScenarioStore, key: str) -> go.Figure:
    """2\u00d72 dashboard: Pred_Units, Incr_Units, GIV, NOS."""
    df = store.load(key)
    panels = [
        ("Pred_Units",  "Predicted Units"),
        ("Incr_Units",  "Incremental Units"),
        ("GIV",         "Gross Invoice Value (AUD)"),
        ("NOS",         "Net Operating Sales (AUD)"),
    ]
    available = [(m, lbl) for m, lbl in panels if m in df.columns]
    fig = make_subplots(rows=2, cols=2, subplot_titles=[lbl for _, lbl in available[:4]])
    palette = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]

    for i, (metric, label) in enumerate(available[:4]):
        row, col = divmod(i, 2)
        row += 1; col += 1
        summary = df.groupby("Scenario Name")[metric].sum().reset_index()
        summary = summary.sort_values(metric, ascending=False)
        colors = [
            "#7f7f7f" if n == "base" else palette[j % len(palette)]
            for j, n in enumerate(summary["Scenario Name"])
        ]
        fig.add_trace(
            go.Bar(x=summary["Scenario Name"], y=summary[metric],
                   marker_color=colors, name=label, showlegend=False),
            row=row, col=col
        )

    fig.update_layout(
        title="What-If Scenario Comparison Dashboard (Coles ANZ)",
        template="plotly_white", height=600, width=950,
    )
    return fig


def plot_lift_waterfall(store: ScenarioStore, key: str,
                        metric: str = "Pred_Units") -> go.Figure:
    """Waterfall chart showing lift vs base."""
    df = store.load(key)
    summary = df.groupby("Scenario Name")[metric].sum().reset_index()
    base_val = summary.loc[summary["Scenario Name"] == "base", metric].values
    if len(base_val) == 0:
        raise ValueError("No 'base' scenario found.")
    base_val = base_val[0]

    non_base = summary[summary["Scenario Name"] != "base"].copy()
    non_base["delta"] = non_base[metric] - base_val
    non_base = non_base.sort_values("delta", ascending=False)

    fig = go.Figure(go.Waterfall(
        name="Lift vs Base", orientation="v",
        x=["Base"] + non_base["Scenario Name"].tolist(),
        y=[base_val] + non_base["delta"].tolist(),
        measure=["absolute"] + ["relative"] * len(non_base),
        textposition="outside",
        text=[f"{int(base_val):,}"] + [f"{int(d):+,}" for d in non_base["delta"]],
        connector={"line": {"color": "#ccc"}},
        increasing={"marker": {"color": "#2ca02c"}},
        decreasing={"marker": {"color": "#d62728"}},
        totals={"marker": {"color": "#1f77b4"}},
    ))
    fig.update_layout(
        title=f"Lift Waterfall \u2014 {metric}",
        template="plotly_white", height=450,
    )
    return fig


print("\u2705 Comparison & visualization functions OK")

✅ Comparison & visualization functions OK


---
## Part 6 — End-to-End What-If Scenario Example

We build **4 scenarios** for **HERBAL ESSENCES** at Coles:

| Scenario | Promotion |
|---|---|
| `base` | No change (historical baseline as-is) |
| `he_50pct_straight` | 50% Straight discount on all Herbal Essences |
| `he_30pct_mb` | 30% Multibuy on all Herbal Essences |
| `he_40pct_straight` | 40% Straight discount on all Herbal Essences |

Then we score, compare, and visualize.

In [11]:
# \u2500\u2500 Step 1: Load data & create base scenario \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
store = ScenarioStore()

raw = filter_zero_base(base_df)
store.save("raw", raw)
copy_base(store, source_key="raw", dest_key="base")
print(f"Base scenario: {len(store.load('base')):,} rows")

# \u2500\u2500 Step 2: Explore data \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
print("\n\u2500\u2500 Available brands & segments \u2500\u2500")
print(get_cols_and_values(store, "raw", columns=["Brand", "Segment", "Category"]))

Base scenario: 12,166 rows

── Available brands & segments ──
{
  "Brand": [
    "HERBAL ESSENCES",
    "HEAD & SHOULDERS",
    "PANTENE",
    "GILLETTE",
    "VENUS"
  ],
  "Segment": [
    "Hair Conditioner",
    "Shampoo",
    "2in1",
    "Hair Treatments",
    "Male Premium Blade",
    "Male Shave Prep",
    "Female Premium Razor",
    "Male Disposable Razor",
    "Female Premium Blade",
    "Female Shave Prep",
    "Male Premium Razor",
    "Female Disposable Razor",
    "Male Fragrance",
    "Male Non-Premium Razor",
    "Female Intimate Grooming",
    "Male Double Edge Blade",
    "Male Intimate Grooming",
    "Male Double Edge Razor"
  ],
  "Category": [
    "HAIR CARE",
    "SHAVE CARE"
  ]
}


In [12]:
# \u2500\u2500 Step 3a: Scenario \u2014 50% Straight on Herbal Essences \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
BRAND = "HERBAL ESSENCES"
filter_criteria = {"Brand": BRAND}

col_filter(store, source_key="raw", filter_criteria=filter_criteria, dest_key="he_rows")

swap_replace(
    store,
    filtered_key="he_rows",
    base_key="base",
    promotion_values={
        "Promo Type": "Discount",
        "Discount Pct": 0.50,
        "Promo Mechanic": "50% Straight",
        "Promo Depth": 1.0,
    },
    dest_key="scenario_50str",
)
rename_scenario(store, "scenario_50str", "he_50pct_straight")
print(f"Scenario 'he_50pct_straight' built: {len(store.load('scenario_50str')):,} rows")

Scenario 'he_50pct_straight' built: 979 rows


In [13]:
# \u2500\u2500 Step 3b: Scenario \u2014 30% Multibuy on Herbal Essences \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
col_filter(store, source_key="raw", filter_criteria=filter_criteria, dest_key="he_rows_mb")

swap_replace(
    store,
    filtered_key="he_rows_mb",
    base_key="base",
    promotion_values={
        "Promo Type": "Multibuy",
        "Discount Pct": 0.30,
        "Promo Mechanic": "30% MB",
        "Promo Depth": 0.0,
        "Is Multibuy": 1,
    },
    dest_key="scenario_30mb",
)
rename_scenario(store, "scenario_30mb", "he_30pct_mb")
print(f"Scenario 'he_30pct_mb' built: {len(store.load('scenario_30mb')):,} rows")

Scenario 'he_30pct_mb' built: 979 rows


In [14]:
# \u2500\u2500 Step 3c: Scenario \u2014 40% Straight on Herbal Essences \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
col_filter(store, source_key="raw", filter_criteria=filter_criteria, dest_key="he_rows_40")

swap_replace(
    store,
    filtered_key="he_rows_40",
    base_key="base",
    promotion_values={
        "Promo Type": "Discount",
        "Discount Pct": 0.40,
        "Promo Mechanic": "40% Straight",
        "Promo Depth": 1.0,
    },
    dest_key="scenario_40str",
)
rename_scenario(store, "scenario_40str", "he_40pct_straight")
print(f"Scenario 'he_40pct_straight' built: {len(store.load('scenario_40str')):,} rows")

Scenario 'he_40pct_straight' built: 979 rows


In [15]:
# \u2500\u2500 Step 4: Concatenate all scenarios \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
concatenate_scenarios(
    store,
    scenario_keys=["scenario_50str", "scenario_30mb", "scenario_40str"],
    base_key="base",
    combined_key="all_scenarios",
    include_base=True,
)

print("\n\u2500\u2500 Scenarios in combined set \u2500\u2500")
list_scenarios_in_df(store, "all_scenarios")


── Scenarios in combined set ──
Scenario Name
base                 979
he_30pct_mb          979
he_40pct_straight    979
he_50pct_straight    979


In [16]:
# \u2500\u2500 Step 5: Score all scenarios \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
mock_score_scenarios(store, key="all_scenarios", scored_key="scored")
scored_df = store.load("scored")
print(f"Scored DataFrame: {len(scored_df):,} rows, {len(scored_df.columns)} columns")
scored_df[["Brand", "SKU Name", "Fiscal Week Start Date", "Scenario Name",
           "Base Consumption Units", "Pred_Units", "Incr_Units", "GIV", "NOS"]].head(8)

Scored 3,916 rows across scenarios: ['base', 'he_50pct_straight', 'he_30pct_mb', 'he_40pct_straight']
Scored DataFrame: 3,916 rows, 36 columns


,Brand,SKU Name,Fiscal Week Start Date,Scenario Name,Base Consumption Units,Pred_Units,Incr_Units,GIV,NOS
87,HERBAL ESSENCES,HE SH REPR ARGAN OIL 4 20.2Z PMP REM,2024-07-03,base,534.10,961,426.90,18259.00,5477.70
88,HERBAL ESSENCES,HE SH REPR ARGAN OIL 4 20.2Z PMP REM,2024-07-10,base,534.10,876,341.90,16644.00,4993.20
89,HERBAL ESSENCES,HE SH REPR ARGAN OIL 4 20.2Z PMP REM,2024-07-17,base,534.10,935,400.90,17765.00,5329.50
90,HERBAL ESSENCES,HE SH REPR ARGAN OIL 4 20.2Z PMP REM,2024-07-24,base,534.10,534,0.00,10146.00,3043.80
91,HERBAL ESSENCES,HE SH REPR ARGAN OIL 4 20.2Z PMP REM,2024-07-31,base,534.10,961,426.90,18259.00,5477.70
92,HERBAL ESSENCES,HE SH REPR ARGAN OIL 4 20.2Z PMP REM,2024-08-07,base,534.10,881,346.90,16739.00,5021.70
93,HERBAL ESSENCES,HE SH REPR ARGAN OIL 4 20.2Z PMP REM,2024-08-14,base,534.10,935,400.90,17765.00,5329.50
94,HERBAL ESSENCES,HE SH REPR ARGAN OIL 4 20.2Z PMP REM,2024-08-21,base,543.80,544,0.20,10336.00,3100.80


In [17]:
# \u2500\u2500 Step 6: Compare scenarios \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
comparison = compare_scenarios(
    store, "scored",
    metrics=["Pred_Units", "Incr_Units", "GIV", "NOS", "EstimatedPromoSpend"]
)
print("\u2500\u2500 Scenario Comparison (aggregated totals) \u2500\u2500")
comparison

── Scenario Comparison (aggregated totals) ──


,Scenario Name,Pred_Units,Incr_Units,GIV,NOS,EstimatedPromoSpend
0,base,908685,211114.28,12495736.00,4339573.39,2566313.32
1,he_30pct_mb,882300,243290.88,12025443.00,3981808.21,2847758.40
2,he_40pct_straight,923357,284347.88,12672400.50,4175895.46,3624145.80
3,he_50pct_straight,923357,284347.88,12672400.50,4175895.46,4206445.95


In [18]:
# \u2500\u2500 Step 7: Lift summary vs base \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
lift_df = get_scenario_lift_summary(store, "scored")
print("\u2500\u2500 Lift % vs Base \u2500\u2500")
lift_df

── Lift % vs Base ──


,Scenario Name,Pred_Units Lift %,Incr_Units Lift %,GIV Lift %,NOS Lift %
0,he_30pct_mb,-2.90,15.24,-3.76,-8.24
1,he_40pct_straight,1.61,34.69,1.41,-3.77
2,he_50pct_straight,1.61,34.69,1.41,-3.77


In [19]:
# \u2500\u2500 Step 8: Visualize \u2014 single metric bar chart \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
fig1 = plot_scenario_comparison(store, "scored", metric="Pred_Units")
fig1.show()

In [20]:
# \u2500\u2500 Step 9: 2\u00d72 Dashboard \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
fig2 = plot_scenario_dashboard(store, "scored")
fig2.show()

In [21]:
# \u2500\u2500 Step 10: Lift waterfall chart \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
fig3 = plot_lift_waterfall(store, "scored", metric="Pred_Units")
fig3.show()

---
## Part 7 — Explore Other Scenarios (Sandbox)

Try different brands, segments, or promo types.

In [22]:
# \u2500\u2500 Example: Compare Pantene promos \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
store2 = ScenarioStore()
store2.save("raw", filter_zero_base(base_df))
copy_base(store2, "raw", "base")

# Scenario A: Pantene 30% Straight
col_filter(store2, "raw", {"Brand": "PANTENE"}, "pantene_rows")
swap_replace(store2, "pantene_rows", "base",
             {"Promo Type": "Discount", "Discount Pct": 0.30,
              "Promo Mechanic": "30% Straight"},
             "pantene_30str")
rename_scenario(store2, "pantene_30str", "pantene_30pct_str")

# Scenario B: Pantene 50% Straight
col_filter(store2, "raw", {"Brand": "PANTENE"}, "pantene_rows2")
swap_replace(store2, "pantene_rows2", "base",
             {"Promo Type": "Discount", "Discount Pct": 0.50,
              "Promo Mechanic": "50% Straight"},
             "pantene_50str")
rename_scenario(store2, "pantene_50str", "pantene_50pct_str")

# Scenario C: Pantene 40% Multibuy
col_filter(store2, "raw", {"Brand": "PANTENE"}, "pantene_rows3")
swap_replace(store2, "pantene_rows3", "base",
             {"Promo Type": "Multibuy", "Discount Pct": 0.40,
              "Promo Mechanic": "40% MB", "Is Multibuy": 1},
             "pantene_40mb")
rename_scenario(store2, "pantene_40mb", "pantene_40pct_mb")

# Combine + Score
concatenate_scenarios(store2,
                      ["pantene_30str", "pantene_50str", "pantene_40mb"],
                      "base", "pantene_all")
mock_score_scenarios(store2, "pantene_all", "pantene_scored")

# Results
print("\u2500\u2500 Pantene Scenario Lift \u2500\u2500")
display(get_scenario_lift_summary(store2, "pantene_scored"))
plot_scenario_dashboard(store2, "pantene_scored").show()

Scored 3,916 rows across scenarios: ['base', 'pantene_30pct_str', 'pantene_50pct_str', 'pantene_40pct_mb']
── Pantene Scenario Lift ──


,Scenario Name,Pred_Units Lift %,Incr_Units Lift %,GIV Lift %,NOS Lift %
0,pantene_30pct_str,31.48,84.60,18.08,11.80
1,pantene_40pct_mb,28.03,69.77,15.07,9.10
2,pantene_50pct_str,33.20,92.02,19.59,13.14


In [23]:
# \u2500\u2500 Example: Cross-brand comparison (same promo) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
# "Which Hair Care brand responds best to 50% Straight?"
store3 = ScenarioStore()
store3.save("raw", filter_zero_base(base_df))
copy_base(store3, "raw", "base")

hair_brands = ["HERBAL ESSENCES", "HEAD & SHOULDERS", "PANTENE"]
scenario_keys = []

for brand in hair_brands:
    fkey = f"{brand.lower().replace(' ', '_')}_rows"
    skey = f"{brand.lower().replace(' ', '_')}_50str"
    col_filter(store3, "raw", {"Brand": brand}, fkey)
    swap_replace(store3, fkey, "base",
                 {"Promo Type": "Discount", "Discount Pct": 0.50,
                  "Promo Mechanic": "50% Straight"},
                 skey)
    rename_scenario(store3, skey, f"{brand.lower().replace(' ', '_')}_50str")
    scenario_keys.append(skey)

concatenate_scenarios(store3, scenario_keys, "base", "brand_comparison")
mock_score_scenarios(store3, "brand_comparison", "brand_scored")

print("\u2500\u2500 Cross-Brand 50% Straight Comparison \u2500\u2500")
display(get_scenario_lift_summary(store3, "brand_scored"))
plot_scenario_dashboard(store3, "brand_scored").show()

Scored 3,916 rows across scenarios: ['base', 'herbal_essences_50str', 'head_&_shoulders_50str', 'pantene_50str']
── Cross-Brand 50% Straight Comparison ──


,Scenario Name,Pred_Units Lift %,Incr_Units Lift %,GIV Lift %,NOS Lift %
0,head_&_shoulders_50str,7.67,34.58,19.04,13.85
1,herbal_essences_50str,1.61,34.69,1.41,-3.77
2,pantene_50str,33.20,92.02,19.59,13.14


In [24]:
# \u2500\u2500 Validate a promotion directly \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
try:
    valid = ColesPromotion(
        promo_type="Multibuy",
        discount_pct=0.40,
        promo_mechanic="40% MB",
    )
    print(f"\u2705 Valid: type={valid.promo_type}, pct={valid.discount_pct}, mb={valid.is_multibuy}")
except Exception as e:
    print(f"\u274c Invalid: {e}")

try:
    invalid = ColesPromotion(
        promo_type="Discount",
        discount_pct=1.5,  # > 1.0 should fail
    )
except Exception as e:
    print(f"\u274c Correctly rejected: {e}")

✅ Valid: type=Multibuy, pct=0.4, mb=1
❌ Correctly rejected: 1 validation error for ColesPromotion
discount_pct
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.5, input_type=float]
    For further information visit https://errors.pydantic.dev/2.12/v/less_than_equal


---
## Part 8 — Single-SKU Scenario Planning Demo

### Business Question
> **We plan to run a 30% off promotion on PTN SHM 900MLX3 DMR ANZ (Pantene Shampoo 900ml 3-pack).
> Compared to 20% off or 40% off under the same conditions, which discount level truly delivers the highest uplift?**

### Approach
1. Extract baseline data for this single SKU
2. Create 3 "what-if" scenarios (20% / 30% / 40% off)
3. Score each scenario (predict units sold)　- using Dummy calculation ()
4. Compare results — **which discount rate is optimal?

### Note
 - Scnearion Operation: Implemented
 - LLM Part: Dummy calculation - blackbox as in SB+ API 

In [46]:
# ============================================================
# Step 1: Identify Target SKU & Prepare Base Data
# ============================================================
TARGET_SKU_NAME = "PTN SHM 900MLX3 DMR ANZ"

# Extract only this 1 SKU and inspect
sku_df = base_df[base_df["SKU Name"] == TARGET_SKU_NAME].copy()
print(f"Target SKU: {TARGET_SKU_NAME}")
print(f"Brand: {sku_df['Brand'].iloc[0]}")
print(f"Category: {sku_df['Category'].iloc[0]}")
print(f"Data points: {len(sku_df)} weeks")
print(f"Date range: {sku_df['Fiscal Week Start Date'].min()} to {sku_df['Fiscal Week Start Date'].max()}")
print(f"\nAvg. Baseline (Base Consumption Units): {sku_df['Base Consumption Units'].mean():.0f} units/week")
print(f"Shelf Price (Non Promo Price): {sku_df['Non Promo Price'].median():.2f} AUD")
print(f"\nHistorical promo mechanics used:")
print(sku_df["Promo Mechanic"].value_counts().to_string())

Target SKU: PTN SHM 900MLX3 DMR ANZ
Brand: PANTENE
Category: HAIR CARE
Data points: 87 weeks
Date range: 2024-07-03 to 2026-02-25

Avg. Baseline (Base Consumption Units): 2427 units/week
Shelf Price (Non Promo Price): 20.00 AUD

Historical promo mechanics used:
Promo Mechanic
40% MB          28
50% Straight    21
2 FOR $24       19
50% OFF         13
Baseline         3
2 FOR $20        2
40% Straight     1


In [42]:
# ============================================================
# Step 2: Create 3 Scenarios (20% / 30% / 40% off)
# ============================================================
sku_store = ScenarioStore()

# Use only rows with baseline > 0
sku_data = filter_zero_base(base_df)
sku_store.save("raw", sku_data)
copy_base(sku_store, "raw", "base")

# Scenario definitions: discount rate & name
SCENARIOS = [
    {"name": "20pct_off", "pct": 0.20, "mechanic": "20% Straight"},
    {"name": "30pct_off", "pct": 0.30, "mechanic": "30% Straight"},
    {"name": "40pct_off", "pct": 0.40, "mechanic": "40% Straight"},
]

scenario_keys = []
for sc in SCENARIOS:
    # 1. Filter to target SKU rows only
    fkey = f"sku_{sc['name']}_rows"
    col_filter(sku_store, "raw", {"SKU Name": TARGET_SKU_NAME}, fkey)

    # 2. Swap promo settings and merge back with base
    skey = f"scenario_{sc['name']}"
    swap_replace(
        sku_store,
        filtered_key=fkey,
        base_key="base",
        promotion_values={
            "Promo Type": "Discount",
            "Discount Pct": sc["pct"],
            "Promo Mechanic": sc["mechanic"],
            "Promo Depth": 1.0,
        },
        dest_key=skey,
    )
    rename_scenario(sku_store, skey, sc["name"])
    scenario_keys.append(skey)
    print(f"Scenario '{sc['name']}' created ({len(sku_store.load(skey)):,} rows)")

print(f"\nScenarios built: {[s['name'] for s in SCENARIOS]}")

Scenario '20pct_off' created (979 rows)
Scenario '30pct_off' created (979 rows)
Scenario '40pct_off' created (979 rows)

Scenarios built: ['20pct_off', '30pct_off', '40pct_off']


In [43]:
# ============================================================
# Step 3: Concatenate Scenarios & Score
# ============================================================
# Combine 3 scenarios + baseline into a single DataFrame
concatenate_scenarios(
    sku_store,
    scenario_keys=scenario_keys,
    base_key="base",
    combined_key="sku_all",
    include_base=True,
)

print("-- Scenarios in combined set --")
list_scenarios_in_df(sku_store, "sku_all")

# Score all scenarios (predict how many units each would sell)
mock_score_scenarios(sku_store, key="sku_all", scored_key="sku_scored")
print("\nScoring complete")

-- Scenarios in combined set --
Scenario Name
20pct_off    979
30pct_off    979
40pct_off    979
base         979
Scored 3,916 rows across scenarios: ['base', '20pct_off', '30pct_off', '40pct_off']

Scoring complete


In [44]:
# ============================================================
# Step 4: Review Scored Results for Target SKU
# ============================================================
scored_df = sku_store.load("sku_scored")

# Filter to this 1 SKU only
sku_scored = scored_df[scored_df["SKU Name"] == TARGET_SKU_NAME].copy()

# If no base rows in scored output, compute base from raw data
if "base" not in sku_scored["Scenario Name"].unique():
    sku_base_raw = sku_store.load("base")
    sku_base_raw = sku_base_raw[sku_base_raw["SKU Name"] == TARGET_SKU_NAME].copy()
    sku_base_raw["Scenario Name"] = "base"
    # Score the base rows using the same uplift logic
    uplifts = sku_base_raw.apply(_compute_row_uplift, axis=1)
    sku_base_raw["Pred_Units"] = (sku_base_raw["Base Consumption Units"] * uplifts).round(0).astype(int)
    sku_base_raw["Incr_Units"] = (sku_base_raw["Pred_Units"] - sku_base_raw["Base Consumption Units"]).clip(lower=0)
    price = sku_base_raw["Non Promo Price"].where(sku_base_raw["Non Promo Price"] > 0, 16.0)
    nos_m = 0.31
    sku_base_raw["GIV"] = (sku_base_raw["Pred_Units"] * price).round(2)
    sku_base_raw["NOS"] = (sku_base_raw["GIV"] * nos_m).round(2)
    discount_pct = sku_base_raw.get("Discount Pct", pd.Series(0, index=sku_base_raw.index)).fillna(0)
    sku_base_raw["EstimatedPromoSpend"] = (discount_pct * price * sku_base_raw["Pred_Units"]).round(2)
    sku_scored = pd.concat([sku_base_raw, sku_scored], ignore_index=True)

print(f"-- Scored results for {TARGET_SKU_NAME} --")
print(f"Rows: {len(sku_scored)}")
print(f"Scenarios: {sku_scored['Scenario Name'].unique().tolist()}\n")

# Aggregate by scenario
sku_comparison = sku_scored.groupby("Scenario Name").agg({
    "Base Consumption Units": "sum",
    "Pred_Units": "sum",
    "Incr_Units": "sum",
    "GIV": "sum",
    "NOS": "sum",
    "EstimatedPromoSpend": "sum",
}).reset_index()

# Calculate uplift %
base_units = sku_comparison.loc[
    sku_comparison["Scenario Name"] == "base", "Pred_Units"
].values[0]
sku_comparison["Uplift vs Base (%)"] = (
    (sku_comparison["Pred_Units"] - base_units) / base_units * 100
).round(1)

# ROI = (Incremental GIV - Promo Spend) / Promo Spend
base_giv = sku_comparison.loc[
    sku_comparison["Scenario Name"] == "base", "GIV"
].values[0]
sku_comparison["ROI"] = (
    (sku_comparison["GIV"] - base_giv - sku_comparison["EstimatedPromoSpend"])
    / sku_comparison["EstimatedPromoSpend"].replace(0, np.nan)
).round(2)

print("-- Scenario Comparison Table --")
sku_comparison

-- Scored results for PTN SHM 900MLX3 DMR ANZ --
Rows: 348
Scenarios: ['base', '20pct_off', '30pct_off', '40pct_off']

-- Scenario Comparison Table --


,Scenario Name,Base Consumption Units,Pred_Units,Incr_Units,GIV,NOS,EstimatedPromoSpend,Uplift vs Base (%),ROI
0,20pct_off,211134.00,316703,105569.00,6334060.00,1963558.60,1266812.00,-2.00,-1.10
1,30pct_off,211134.00,369444,158310.00,7388880.00,2290552.80,2216664.00,14.40,-0.58
2,40pct_off,211134.00,380053,168919.00,7601060.00,2356328.60,3040424.00,17.60,-0.63
3,base,211134.00,323059,111925.00,6461180.00,2002965.80,2412748.00,0.00,-1.00


In [45]:
# ============================================================
# Step 5: Visualization — 3-Scenario Comparison Charts
# ============================================================
# Chart only the promo scenarios (exclude base)
sc_data = sku_comparison[sku_comparison["Scenario Name"] != "base"].copy()
sc_names = sc_data["Scenario Name"].values
colors_3 = ["#3498db", "#2ecc71", "#e67e22"]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Predicted Units (Pred_Units)",
        "Incremental Units (Incr_Units)",
        "Uplift vs Base (%)",
        "Promo Spend vs Incremental GIV",
    ],
)

# 1. Pred_Units
fig.add_trace(go.Bar(
    x=sc_names, y=sc_data["Pred_Units"],
    marker_color=colors_3, text=sc_data["Pred_Units"].astype(int),
    textposition="outside", showlegend=False,
), row=1, col=1)
# Baseline reference line
fig.add_hline(y=base_units, line_dash="dash", line_color="gray",
              annotation_text=f"Base: {int(base_units):,}", row=1, col=1)

# 2. Incr_Units
fig.add_trace(go.Bar(
    x=sc_names, y=sc_data["Incr_Units"],
    marker_color=colors_3, text=sc_data["Incr_Units"].astype(int),
    textposition="outside", showlegend=False,
), row=1, col=2)

# 3. Uplift %
fig.add_trace(go.Bar(
    x=sc_names, y=sc_data["Uplift vs Base (%)"],
    marker_color=colors_3,
    text=[f"{v:.1f}%" for v in sc_data["Uplift vs Base (%)"]],
    textposition="outside", showlegend=False,
), row=2, col=1)

# 4. Spend vs Incremental GIV (grouped bar)
base_giv = sku_comparison.loc[sku_comparison["Scenario Name"] == "base", "GIV"].values[0]
sc_data_copy = sc_data.copy()
sc_data_copy["Incr_GIV"] = sc_data_copy["GIV"] - base_giv
fig.add_trace(go.Bar(
    x=sc_names, y=sc_data_copy["Incr_GIV"],
    name="Incremental GIV", marker_color="#2ecc71",
), row=2, col=2)
fig.add_trace(go.Bar(
    x=sc_names, y=sc_data_copy["EstimatedPromoSpend"],
    name="Promo Spend", marker_color="#e74c3c",
), row=2, col=2)

fig.update_layout(
    title=f"What-If Scenario Comparison: {TARGET_SKU_NAME}<br>"
          f"<sub>20% off vs 30% off vs 40% off — Which discount rate is optimal?</sub>",
    template="plotly_white",
    height=650, width=950,
    barmode="group",
)
fig.show()

---
## Recommendation — Optimal Promotion Strategy for PTN SHM 900MLX3 DMR ANZ

In [35]:
# ============================================================
# Final Recommendation — Data-Driven Summary
# ============================================================

# Identify the best scenario (highest ROI among promo scenarios)
promo_scenarios = sku_comparison[sku_comparison["Scenario Name"] != "base"].copy()
best = promo_scenarios.loc[promo_scenarios["ROI"].idxmax()]
worst = promo_scenarios.loc[promo_scenarios["ROI"].idxmin()]

best_sd = best['EstimatedPromoSpend'] / best['GIV'] * 100

print("=" * 65)
print(f"  ★ RECOMMENDED PROMOTION PLAN")
print("=" * 65)
print(f"""
  SKU          : {TARGET_SKU_NAME}
  Brand        : PANTENE
  Category     : HAIR CARE
  Shelf Price  : {sku_df['Non Promo Price'].median():.2f} AUD
  Baseline     : {sku_df['Base Consumption Units'].mean():,.0f} units/week avg

  ─────────────────────────────────────────────────────────
  Recommended  : {best['Scenario Name'].upper()}
  Pred. Units  : {int(best['Pred_Units']):>12,}
  Incr. Units  : {int(best['Incr_Units']):>12,}
  GIV (Revenue): ${best['GIV']:>14,.0f}
  Promo Spend  : ${best['EstimatedPromoSpend']:>14,.0f}
  Uplift       : {best['Uplift vs Base (%)']:>+.1f}% vs baseline
  SD%GIV       : {best_sd:>.1f}%
  ROI          : {best['ROI']:>+.2f}
  ─────────────────────────────────────────────────────────
""")

for i, (_, row) in enumerate(promo_scenarios.iterrows()):
    name = row["Scenario Name"]
    sd_giv = row['EstimatedPromoSpend'] / row['GIV'] * 100 if row['GIV'] != 0 else 0
    label = chr(ord('A') + i)
    print(f"\n  ─────────────────────────────────────────────────────────")
    print(f"  Plan {label}        : {name.upper()}")
    print(f"  Pred. Units  : {int(row['Pred_Units']):>12,}")
    print(f"  Incr. Units  : {int(row['Incr_Units']):>12,}")
    print(f"  GIV (Revenue): ${row['GIV']:>14,.0f}")
    print(f"  Promo Spend  : ${row['EstimatedPromoSpend']:>14,.0f}")
    print(f"  Uplift       : {row['Uplift vs Base (%)']:>+.1f}% vs baseline")
    print(f"  SD%GIV       : {sd_giv:>.1f}%")
    print(f"  ROI          : {row['ROI']:>+.2f}")

print("\n" + "=" * 65)
print("  SCENARIO COMPARISON (all 3 scenarios)")
print("=" * 65)

# Build comparison table for Uplift & SD%GIV
print(f"\n  {'Plan':<12} {'Scenario':<16} {'Uplift':>10} {'SD%GIV':>10} {'ROI':>8}")
print(f"  {'─'*12} {'─'*16} {'─'*10} {'─'*10} {'─'*8}")
for i, (_, row) in enumerate(promo_scenarios.iterrows()):
    label = f"Plan {chr(ord('A') + i)}"
    name = row['Scenario Name'].upper()
    uplift = row['Uplift vs Base (%)']
    sd = row['EstimatedPromoSpend'] / row['GIV'] * 100 if row['GIV'] != 0 else 0
    roi = row['ROI']
    marker = " ★" if row['Scenario Name'] == best['Scenario Name'] else ""
    print(f"  {label:<12} {name:<16} {uplift:>+9.1f}% {sd:>9.1f}% {roi:>+7.2f}{marker}")

print(f"""
  ─────────────────────────────────────────────────────────
  ★ = Recommended plan (best ROI)
  SD%GIV = Promo Spend / GIV × 100
  Lower SD%GIV = more efficient promo investment.
  ─────────────────────────────────────────────────────────
""")

  ★ RECOMMENDED PROMOTION PLAN

  SKU          : PTN SHM 900MLX3 DMR ANZ
  Brand        : PANTENE
  Category     : HAIR CARE
  Shelf Price  : 20.00 AUD
  Baseline     : 2,427 units/week avg

  ─────────────────────────────────────────────────────────
  Recommended  : 30PCT_OFF
  Pred. Units  :      369,444
  Incr. Units  :      158,310
  GIV (Revenue): $     7,388,880
  Promo Spend  : $     2,216,664
  Uplift       : +14.4% vs baseline
  SD%GIV       : 30.0%
  ROI          : -0.58
  ─────────────────────────────────────────────────────────


  ─────────────────────────────────────────────────────────
  Plan A        : 20PCT_OFF
  Pred. Units  :      316,703
  Incr. Units  :      105,569
  GIV (Revenue): $     6,334,060
  Promo Spend  : $     1,266,812
  Uplift       : -2.0% vs baseline
  SD%GIV       : 20.0%
  ROI          : -1.10

  ─────────────────────────────────────────────────────────
  Plan B        : 30PCT_OFF
  Pred. Units  :      369,444
  Incr. Units  :      158,310
  GIV (R